# Preprocessing Pipeline — DeepLabCut vs. TopScan
**Memory Laboratory - Brain Institute (UFRN) | Technical Validation of Animal Tracking**

---

## Overview

This notebook performs raw tracking data preprocessing before statistical analysis.
It must be run **before** the notebook `Analysis Pipeline — DeepLabCut vs. TopScan.ipynb`.

**Pipeline flow:**
```
Videos .MPG  →  Conversion .MP4  →  Crop per animal
     ↓
Files .H5 (DLC)  →  Likelihood Filter (> 0.9)  →  TXT per bodypart
     ↓
TXT TopScan (raw)  →  Remove -1  →  IQR + DBSCAN  →  Savitzky-Golay  →  Calibration (Regression)  →  Final TXT
```

---

## How to Run

Execute cells **in order, top to bottom**. Each section is independent after Drive mount.

| Section | What it does | Needs configuration? |
|---|---|---|
| Dependency Installation | Installs ffmpeg, scipy, scikit-learn | No |
| Video Conversion | .MPG → .MP4 with ffmpeg | Yes — `input_folder`, `output_folder`, `initial` |
| Video Cropping | Splits multi-animal video by quadrant | Yes — `videos_folder` |
| H5 → TXT Export | Extracts DLC coordinates with likelihood filter | Yes — `experiment`, `bodypart_selected` |
| TopScan Calibration | Calibrates, filters outliers and merges events | Yes — `base_path`, `output_path_dir` |

---

## Parameters You Must Adjust

```python
# --- DLC Section ---
experiment           = "experiment"    # Prefix of the .h5 files
bodypart_selected    = 'tail_tip'      # Bodypart of interest (see bodypart_mapping)
LIKELIHOOD_THRESHOLD = 0.9             # Frames below this become NaN

# --- TopScan Section ---
base_path        = "...original TXT and XLSX/"  # Folder with raw .TXT and .XLSX files
output_path_dir  = "...new TXT/"                # Output folder for calibrated files
```

> **Tip:** Never set `LIKELIHOOD_THRESHOLD` below `0.85`.
> Predictions with low likelihood indicate occlusion or ResNet-50 model failure.

---

## Expected Google Drive Structure

```
MyDrive/
└── validacao_topscan/
    ├── DEEPLABCUT/
    │   ├── h5_validacao/          ← .h5 files exported by DLC
    │   └── trajetoria_txt_dlc/    ← TXTs generated by this notebook
    └── TOPSCAN/
        ├── TXT original e XLSX original/  ← raw TopScan files
        └── TXT novo/                      ← calibrated TXTs generated by this notebook
```

---

## About the Applied Filters

- **IQR (displacement):** Removes physically impossible frame-to-frame jumps. Default threshold: `Q3 + 2.5 * IQR`.
- **DBSCAN:** Removes isolated prediction clusters (glitches). Parameter `eps=15 px` — adjust if video resolution differs greatly from ~40 px/cm.
- **Savitzky-Golay:** Smooths high-frequency noise without distorting velocity peaks. Default window: `11 frames` (~0.37 s at 30 fps).
- **Huber Regression:** Calibrates TopScan → DLC space with intercept (corrects translation). R² < 0.90 indicates lens distortion — report to the person responsible for camera setup.

---

#CONNECT TO YOUR DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##DEPENDENCY INSTALLATION

In [ ]:
!apt-get install -q ffmpeg
!pip install -q ffmpeg-python moviepy
!pip install -q --upgrade pandas scikit-learn scipy

##IMPORTS

In [ ]:
import os
import re
import subprocess

import cv2
import ffmpeg
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import r2_score
from sklearn.cluster import DBSCAN
from tqdm import tqdm

##VIDEO CONVERSION: .MPG TO .MP4 (TOPSCAN)

In [ ]:
input_folder  = '/content/drive/My Drive/TESTES_MPG'
output_folder = '/content/drive/My Drive/TESTES_MP4'
initial = ('E9L3D2TR', 'E9L3D3TR')

os.makedirs(output_folder, exist_ok=True)

def convert_videos(input_folder, output_folder):
    files = [f for f in os.listdir(input_folder)
             if f.lower().endswith('.mpg') and f.startswith(initial)]

    with tqdm(total=len(files), desc="Converting videos", unit="video") as pbar:
        for filename in files:
            input_path  = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, os.path.splitext(filename)[0] + '.mp4')

            try:
                try:
                    duration = float(ffmpeg.probe(input_path)['format']['duration'])
                except ffmpeg.Error as e:
                    print(f'\nError reading {filename}: {e.stderr.decode("utf-8")}')
                    pbar.update(1)
                    continue

                command = ['ffmpeg', '-i', input_path, '-r', '30', output_path]
                process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

                last_progress = -1
                while True:
                    line = process.stderr.readline()
                    if not line and process.poll() is not None:
                        break
                    if "time=" in line:
                        match = re.search(r'time=(\d+:\d+:\d+\.\d+)', line)
                        if match:
                            current_time = sum(float(x) * 60 ** i for i, x in enumerate(reversed(match.group(1).split(':'))))
                            progress = (current_time / duration) * 100
                            if int(progress) != last_progress:
                                print(f'\rConverting {filename}: {int(progress)}% complete', end='')
                                last_progress = int(progress)

                process.wait()
                print(f'\nConverted {filename}')
                pbar.update(1)

            except Exception as e:
                print(f'\nError converting {filename}: {e}\n')
                pbar.update(1)

convert_videos(input_folder, output_folder)
print("Conversion complete!")

Convertendo vídeos:   0%|          | 0/2 [00:00<?, ?vídeo/s]

Convertendo E9L3D3TR.MPG: 99% concluído

Convertendo vídeos:  50%|█████     | 1/2 [3:20:44<3:20:44, 12044.36s/vídeo]


Convertido E9L3D3TR.MPG para E9L3D3TR.mp4

Convertendo E9L3D2TR.MPG: 99% concluído

Convertendo vídeos: 100%|██████████| 2/2 [8:31:10<00:00, 15335.15s/vídeo]


Convertido E9L3D2TR.MPG para E9L3D2TR.mp4

Conversão concluída!


##MULTI-ANIMAL VIDEO CROPPING

In [ ]:
def crop_and_save_with_bar(video_path, output_base, bar_width=35):
    filename         = os.path.basename(video_path)
    name_no_ext      = os.path.splitext(filename)[0]
    ids              = name_no_ext.split("-")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening: {video_path}")
        return

    fps          = cap.get(cv2.CAP_PROP_FPS)
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    codec        = cv2.VideoWriter_fourcc(*'mp4v')

    coords = [
        (35, 0,                   frame_width // 2 + 5,  frame_height // 2),
        (frame_width // 2 + 30,   0,                      frame_width - 20,  frame_height // 2),
        (35, frame_height // 2 + 5, frame_width // 2 + 5, frame_height - 10),
    ]

    outputs = {}
    for idx in range(min(len(ids), len(coords))):
        x1, y1, x2, y2 = coords[idx]
        output_path     = os.path.join(output_base, f"{ids[idx].strip()}.mp4")
        outputs[idx]    = cv2.VideoWriter(output_path, codec, fps, (x2 - x1, y2 - y1))

    bar_color = (57, 60, 60)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        for idx, writer in outputs.items():
            x1, y1, x2, y2 = coords[idx]
            crop = frame[y1:y2, x1:x2].copy()
            crop[:, -bar_width:] = bar_color
            writer.write(crop)

    cap.release()
    for writer in outputs.values():
        writer.release()
    print(f"Done: {filename} -> {len(outputs)} videos generated.")

videos_folder = "/content/drive/MyDrive/videos_raquel"
output_base   = os.path.join(videos_folder, "videos_processados")
os.makedirs(output_base, exist_ok=True)

for arquivo in os.listdir(videos_folder):
    if arquivo.endswith(".mp4") and "-" in arquivo:
        crop_and_save_with_bar(os.path.join(videos_folder, arquivo), output_base)

✔️ Finalizado: 1-2.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 3-4.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 5-6.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 7-8.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 9-10.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 11-12.mp4 -> Gerados 2 vídeos.


##H5 → TXT EXPORT (DEEPLABCUT) WITH LIKELIHOOD FILTER

In [ ]:
# =================================================================
# USER SETTINGS
# =================================================================
experiment           = "experimento"
bodypart_selected    = 'tail_tip'   # CHANGE THIS
LIKELIHOOD_THRESHOLD = 0.9          # frames below this become NaN
# =================================================================

bodypart_mapping = {
    'nose': 'nose', 'neck': 'neck', 'centerbody': 'centerbody',
    'tail_base': 'tail_base', 'tail_middle': 'tail_middle', 'tail_tip': 'tail_tip',
}

h5_directory  = '/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/h5_validacao'
txt_directory = '/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/'
os.makedirs(txt_directory, exist_ok=True)

def extract_valid_rat_ids(filename):
    match = re.search(r'experimento(.*?)DLC', filename, re.IGNORECASE)
    return re.findall(r'\d+', match.group(1))[:1] if match else []

for arquivo in [f for f in os.listdir(h5_directory) if f.lower().startswith(experiment) and f.lower().endswith(".h5")]:
    h5_path = os.path.join(h5_directory, arquivo)
    rat_ids = extract_valid_rat_ids(arquivo)
    print(f"\nProcessing: {arquivo} | ID: {rat_ids}")

    try:
        df            = pd.read_hdf(h5_path, header=[0, 1])
        num_frames    = len(df)
        base_column   = df.columns.levels[0][0]
        bodypart_name = bodypart_mapping.get(bodypart_selected, bodypart_selected)

        x          = df[base_column][bodypart_selected]['x']
        y          = df[base_column][bodypart_selected]['y']
        likelihood = df[base_column][bodypart_selected]['likelihood']

        # Confidence filter: coordinates below threshold become NaN
        low_conf_mask = likelihood < LIKELIHOOD_THRESHOLD
        x = x.where(~low_conf_mask, other=np.nan)
        y = y.where(~low_conf_mask, other=np.nan)

        n_rej = int(low_conf_mask.sum())
        print(f"  Likelihood < {LIKELIHOOD_THRESHOLD}: {n_rej} frames rejected ({n_rej/num_frames*100:.1f}%)")
        print(f"  Valid frames exported: {num_frames - n_rej}")

        rat_id   = rat_ids[0] if rat_ids else "X"
        txt_name = f"{experiment}_R{rat_id}_trajetoria_{bodypart_name}.txt"

        pd.DataFrame({
            'Frame': range(num_frames),
            'X': x.round(2),
            'Y': y.round(2),
        }).to_csv(
            os.path.join(txt_directory, txt_name),
            sep=' ', index=False, float_format='%.2f', na_rep='NaN',
        )
        print(f"  {txt_name} saved.")

    except Exception as e:
        print(f"Error during processing: {e}")

print("\nExport complete!")


Processando: experimento11DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['11']
  ✅ experimento_R11_trajetoria_tail_tip.txt

Processando: experimento5DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['5']
  ✅ experimento_R5_trajetoria_tail_tip.txt

Processando: experimento7DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['7']
  ✅ experimento_R7_trajetoria_tail_tip.txt

Processando: experimento13DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['13']
  ✅ experimento_R13_trajetoria_tail_tip.txt

Processando: experimento18DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['18']
  ✅ experimento_R18_trajetoria_tail_tip.txt

Processando: experimento16DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['16']
  ✅ experimento_R16_trajetoria_tail_tip.txt

Processando: experimento2DLC_resnet50_validacao_topscan-dlcMay2

##PREPROCESSING FUNCTIONS: OUTLIERS AND SMOOTHING

In [ ]:
def filter_trajectory_outliers(df, col_x='X', col_y='Y', eps=15.0, min_samples=3, iqr_multiplier=2.5):
    """
    Removes outliers using two complementary methods:
    1. IQR on frame-to-frame displacement (physically impossible jumps).
    2. DBSCAN on the remaining valid coordinates (isolated prediction glitches).

    Parameters for rodents in arenas (~40x40 cm, 30 fps):
      eps=15 px  → maximum distance between points in the main cluster. Adjust according to px/cm of the video.
      iqr_multiplier=2.5 → more conservative than 1.5 to preserve real escape accelerations.
    """
    df = df.copy()

    # --- Step 1: IQR on displacement ---
    dx = np.diff(df[col_x].fillna(method='ffill').values, prepend=df[col_x].iloc[0])
    dy = np.diff(df[col_y].fillna(method='ffill').values, prepend=df[col_y].iloc[0])
    displacement = np.sqrt(dx**2 + dy**2)

    Q1, Q3  = np.nanpercentile(displacement, 25), np.nanpercentile(displacement, 75)
    threshold = Q3 + iqr_multiplier * (Q3 - Q1)
    jumps   = displacement > threshold
    df.loc[jumps, [col_x, col_y]] = np.nan
    print(f"  IQR: {jumps.sum()} jumps removed (threshold: {threshold:.1f} px/frame)")

    # --- Step 2: DBSCAN on valid coordinates ---
    valid_idx    = df[[col_x, col_y]].dropna().index
    valid_coords = df.loc[valid_idx, [col_x, col_y]].values

    if len(valid_coords) > min_samples:
        labels        = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(valid_coords)
        main_cluster  = pd.Series(labels[labels != -1]).mode()[0] if any(labels != -1) else 0
        noise_idx     = valid_idx[labels != main_cluster]
        df.loc[noise_idx, [col_x, col_y]] = np.nan
        n_noise = int((labels == -1).sum() + (labels != main_cluster).sum())
        print(f"  DBSCAN: {n_noise} isolated points removed")

    return df


def smooth_trajectory(df, col_x='X', col_y='Y', window=11, order=3):
    """
    Savitzky-Golay filter applied after linear interpolation of internal NaNs.

    window=11 frames (~0.37 s at 30 fps): preserves exploration and escape accelerations.
    order=3: cubic polynomial — captures curvature without flattening velocity peaks.
    Windows >21 frames tend to distort sudden stops and U-turns.
    """
    df = df.copy()
    for col in [col_x, col_y]:
        series       = df[col].copy()
        series_interp = series.interpolate(method='linear', limit_direction='forward', limit=5)
        n_valid      = series_interp.notna().sum()

        if n_valid >= window:
            sg_values = savgol_filter(
                series_interp.fillna(method='ffill').fillna(method='bfill'),
                window_length=window, polyorder=order,
            )
            # Restore original NaNs — do not mask missing data with smoothed values
            df[col] = np.where(series.isna(), np.nan, sg_values)
        else:
            print(f"  Warning: {col} has only {n_valid} valid points — SG not applied.")

    return df

--- Iniciando Processo Automático de Calibração e Mesclagem do TopScan ---

--- Processando: RATO_10_TCR ---
Iniciando processo para RATO_10_TCR.TXT...
  Dados de trajetória TopScan limpos: 8936 linhas válidas encontradas.
  Fator de Escala X (TS/DLC) calculado: 4.8437
  Fator de Escala Y (TS/DLC) calculado: 3.8533
  Calibração aplicada com sucesso ao TopScan.
  ✅ Arquivo calibrado e mesclado salvo em: /content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/RATO_10_TCR_NOVO.TXT

--- Processando: RATO_12_TCR ---
Iniciando processo para RATO_12_TCR.TXT...
  Dados de trajetória TopScan limpos: 8971 linhas válidas encontradas.
  Fator de Escala X (TS/DLC) calculado: 5.2125
  Fator de Escala Y (TS/DLC) calculado: 11.6934
  Calibração aplicada com sucesso ao TopScan.
  ✅ Arquivo calibrado e mesclado salvo em: /content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/RATO_12_TCR_NOVO.TXT

--- Processando: RATO_15_TCR ---
Iniciando processo para RATO_15_TCR.TXT...
  Dados de trajetória TopSc

##TOPSCAN TXT CREATION WITH REGRESSION CALIBRATION

In [ ]:
# --- 1. Directory Configuration ---
base_path         = "/content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT original e XLSX original/"
output_path_dir   = "/content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/"
dlc_ref_directory = "/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/"


# --- 2. Temporal Alignment Diagnostics ---
def diagnose_temporal_alignment(df_ts, df_dlc, col_frame='FrameNum'):
    """
    Verifies temporal integrity before merge:
    - Duplicate frames and gaps in each series.
    - Overlap percentage between TS and DLC.
    - Offset estimate via cross-correlation when overlap < 85%.
    """
    print("\n  --- Temporal Diagnostics ---")
    for name, df in [('TopScan', df_ts), ('DLC', df_dlc)]:
        frames = df[col_frame].sort_values()
        n_dupl = int(frames.duplicated().sum())
        gaps   = frames.diff().dropna()
        n_gaps = int((gaps > 1).sum())
        if n_dupl:
            print(f"  {name}: {n_dupl} DUPLICATE frames")
        if n_gaps:
            print(f"  {name}: {n_gaps} gaps in frame sequence")
        if not n_dupl and not n_gaps:
            print(f"  {name}: continuous sequence ({int(frames.min())}–{int(frames.max())})")

    frames_ts  = set(df_ts[col_frame])
    frames_dlc = set(df_dlc[col_frame])
    overlap    = len(frames_ts & frames_dlc)
    pct        = overlap / max(len(frames_ts), len(frames_dlc)) * 100
    print(f"  Overlap: {overlap} frames ({pct:.1f}%) | TS={len(frames_ts)}, DLC={len(frames_dlc)}")

    if pct < 85:
        print("  Warning: overlap < 85% — investigate temporal offset.")
        ts_sorted  = df_ts.sort_values(col_frame).set_index(col_frame)
        dlc_sorted = df_dlc.sort_values(col_frame).set_index(col_frame)
        common_idx = ts_sorted.index.intersection(dlc_sorted.index)
        if len(common_idx) > 50:
            from numpy.fft import fft, ifft
            signal_ts  = np.sqrt(ts_sorted.loc[common_idx, 'CenterX(mm)'].diff()**2 +
                                 ts_sorted.loc[common_idx, 'CenterY(mm)'].diff()**2).fillna(0).values
            signal_dlc = np.sqrt(dlc_sorted.loc[common_idx, 'X_dlc'].diff()**2 +
                                 dlc_sorted.loc[common_idx, 'Y_dlc'].diff()**2).fillna(0).values
            correlation      = np.real(ifft(fft(signal_ts) * np.conj(fft(signal_dlc))))
            estimated_offset = int(np.argmax(correlation[:len(correlation) // 2]))
            print(f"  Estimated offset via cross-correlation: ~{estimated_offset} frames")

    return overlap


# --- 3. Calibration via Robust Linear Regression (Huber) ---
def calibrate_by_regression(df_clean_ts, df_dlc_ref,
                             col_ts_x='CenterX(mm)', col_ts_y='CenterY(mm)'):
    """
    Maps TopScan coordinates -> DLC space via Huber regression (robust to outliers).
    Includes intercept — corrects translation beyond scaling.
    R² < 0.90 indicates lens distortion or non-linear camera/arena misalignment.
    """
    df_merged = pd.merge(df_clean_ts, df_dlc_ref, on='FrameNum', how='inner').dropna()

    if len(df_merged) < 30:
        print("  Warning: insufficient points for regression. Skipping calibration.")
        return df_clean_ts.copy(), None

    diagnose_temporal_alignment(df_clean_ts, df_dlc_ref)

    df_calibrated = df_clean_ts.copy()
    models        = {}

    for axis_ts, axis_dlc in [(col_ts_x, 'X_dlc'), (col_ts_y, 'Y_dlc')]:
        X       = df_merged[[axis_ts]].values
        y_target = df_merged[axis_dlc].values

        model = HuberRegressor(epsilon=1.5).fit(X, y_target)
        r2    = r2_score(y_target, model.predict(X))

        print(f"  Regression {axis_ts}: coef={model.coef_[0]:.5f}, intercept={model.intercept_:.3f}, R²={r2:.4f}")
        if r2 < 0.90:
            print(f"  Warning: R²={r2:.3f} low for {axis_ts}. Check camera/arena alignment.")

        df_calibrated[axis_ts] = model.coef_[0] * df_calibrated[axis_ts] + model.intercept_
        models[axis_ts] = {'coef': model.coef_[0], 'intercept': model.intercept_, 'r2': r2}

    return df_calibrated, models


# --- 4. Main Processing Function ---
def process_and_calibrate_topscan(txt_path, xlsx_path, output_path):
    print(f"\nStarting process for {os.path.basename(txt_path)}...")

    # Read original TopScan TXT
    try:
        txt_columns      = ['FrameNum', 'CenterX(mm)', 'CenterY(mm)', 'Areas']
        df_trajectory_ts = pd.read_csv(txt_path, sep=r'\s+', skiprows=1, names=txt_columns)
        df_trajectory_ts['Areas'] = df_trajectory_ts['Areas'].str.replace(',', '', regex=False)
    except Exception as e:
        print(f"  ERROR reading TopScan TXT: {e}")
        return

    df_trajectory_ts['CenterX(mm)'] = pd.to_numeric(df_trajectory_ts['CenterX(mm)'], errors='coerce')
    df_trajectory_ts['CenterY(mm)'] = pd.to_numeric(df_trajectory_ts['CenterY(mm)'], errors='coerce')

    # Remove TopScan "animal not detected" flag (-1)
    df_clean_ts = df_trajectory_ts[
        (df_trajectory_ts['CenterX(mm)'] != -1) & (df_trajectory_ts['CenterY(mm)'] != -1)
    ].copy()
    print(f"  TopScan: {len(df_clean_ts)} rows after removing -1.")

    # Filter outliers (IQR + DBSCAN)
    df_clean_ts = filter_trajectory_outliers(df_clean_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')

    # Kinematic smoothing (Savitzky-Golay)
    df_clean_ts = smooth_trajectory(df_clean_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')

    # Regression calibration
    match = re.search(r'RATO_(\d+)', os.path.basename(txt_path))
    if not match:
        print("  Warning: rat ID not found in filename. Skipping calibration.")
        df_calibrated_ts = df_clean_ts.copy()
    else:
        rat_id       = match.group(1)
        dlc_ref_path = os.path.join(dlc_ref_directory, f"experimento_R{rat_id}_trajetoria_centerbody.txt")

        if not os.path.exists(dlc_ref_path):
            print(f"  Warning: DLC reference not found at '{dlc_ref_path}'. Skipping calibration.")
            df_calibrated_ts = df_clean_ts.copy()
        else:
            try:
                df_dlc_ref = pd.read_csv(dlc_ref_path, sep=r'\s+').rename(
                    columns={'Frame': 'FrameNum', 'X': 'X_dlc', 'Y': 'Y_dlc'}
                )
                # Remove DLC frames with NaN (likelihood < threshold exported as NaN)
                df_dlc_ref = df_dlc_ref.dropna(subset=['X_dlc', 'Y_dlc'])

                df_calibrated_ts, _ = calibrate_by_regression(df_clean_ts, df_dlc_ref)
            except Exception as e:
                print(f"  ERROR during calibration: {e}")
                df_calibrated_ts = df_clean_ts.copy()

    # Read Excel for events
    try:
        df_events = pd.read_excel(xlsx_path, header=1)
        df_events.columns = df_events.columns.str.strip()
    except Exception as e:
        print(f"  ERROR reading Excel: {e}")
        return

    # Assign events to the Areas column
    df_merged_final = df_calibrated_ts.copy()
    df_merged_final['Areas'] = 'Floor'

    for _, row in df_events.iterrows():
        try:
            frame_start = int(row['From Frame'])
            frame_end   = int(row['To Frame'])
            event       = str(row['Event'])
            clean_event = event.split("sniffing On")[-1].strip() if "sniffing On" in event else event.strip()
            df_merged_final.loc[
                (df_merged_final['FrameNum'] >= frame_start) &
                (df_merged_final['FrameNum'] <= frame_end), 'Areas'
            ] = clean_event
        except (KeyError, ValueError) as e:
            print(f"  Warning: problem processing Excel row: {e}. Skipping.")
            continue

    # Save final TXT
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f_out:
        f_out.write("FrameNum CenterX(mm) CenterY(mm) Areas\n")
        for _, row in df_merged_final.iterrows():
            f_out.write(f"{int(row['FrameNum'])} {row['CenterX(mm)']:.2f} {row['CenterY(mm)']:.2f} {row['Areas']}\n")
    print(f"  File saved to: {output_path}")


# --- 5. Automatic Execution ---
def run_automatic_process():
    print("--- Starting TopScan Calibration and Merge ---")
    if not os.path.isdir(base_path):
        print(f"ERROR: base directory '{base_path}' not found.")
        return

    base_names = sorted({os.path.splitext(f)[0] for f in os.listdir(base_path) if f.upper().endswith('.TXT')})

    for base_name in base_names:
        txt_path_current  = os.path.join(base_path, f"{base_name}.TXT")
        xlsx_path_current = os.path.join(base_path, f"{base_name}.xlsx")
        output_path_current = os.path.join(output_path_dir, f"{base_name}_NOVO.TXT")

        if os.path.exists(txt_path_current) and os.path.exists(xlsx_path_current):
            process_and_calibrate_topscan(txt_path_current, xlsx_path_current, output_path_current)
        else:
            print(f"  Warning: TXT or XLSX missing for '{base_name}'. Skipping.")

    print("\n--- Process complete ---")

run_automatic_process()